# Removing all eddies that have land mass in 3 eddy radii (3R)

removing all eddies with NaNs in SST -> indication for landmass

In [1]:
import pandas as pd
import intake
import numpy as np
import matplotlib.pylab as plt
import xarray as xr
import matplotlib.colors as colors
from shapely.geometry import Polygon, Point, box
from shapely import contains_xy
import time
import shapely
from shapely.affinity import scale

In [2]:
#import data with ice free in 3R
edso_ac_0 = xr.open_dataset("/scratch/b/b383696/eddy_data/oceanS60/edso_ac_3R_0.nc")
edso_c_0 = xr.open_dataset("/scratch/b/b383696/eddy_data/oceanS60/edso_c_3R_0.nc")

In [3]:
eerie_cat=intake.open_catalog("https://raw.githubusercontent.com/eerie-project/intake_catalogues/main/eerie.yaml")
da_atmos = eerie_cat["dkrz"]["disk"]["model-output"]["icon-esm-er"]["hist-1950"]["v20240618"]["atmos"]["gr025"]["2d_daily_mean"].to_dask()
da_ocean = eerie_cat["dkrz"]["disk"]["model-output"]["icon-esm-er"]["hist-1950"]["v20240618"]["ocean"]["gr025"]["2d_daily_mean"].to_dask()

In [4]:
#select atmos and ocean data for SO (puffer zone of 10°)
da_atmos_so = da_atmos.where(da_atmos.lat < -50, drop = True)
da_ocean_so = da_ocean.where(da_ocean.lat < -50, drop = True)

In [7]:
#mask for 3R for ac eddies
#starting timer
start_time = time.time()
    
#select variable conc for ocean
da_ocean_sst = da_ocean["to"].squeeze()


def dlon(lon, lon0):
    return ((lon - lon0 + 180) % 360) - 180

#creating new data.frame with only the effective_contour_lat/lon and time of the eddies
dt_contour_ac = xr.Dataset(
        {
            "contour_lat": edso_ac_0["effective_contour_latitude"],
            "contour_lon": edso_ac_0["effective_contour_longitude"],
            "time" : edso_ac_0["time"],
            "radius" : edso_ac_0["effective_radius"],
            "lat" : edso_ac_0["latitude"],
            "lon" : edso_ac_0["longitude"],
            "ID" : edso_ac_0["ID"]
        }
    )

n_obs_ac = dt_contour_ac.sizes["obs"]
    
#changing radius in m to lat/lon, using simplification 
# 1° lat = 111.32 km = 111320 m
# 1° lon = 111.32 km * cos(lat of eddy cetnre) = 111320 m * cos(lat of eddy centre)
    
radius_lat = dt_contour_ac["radius"].values / 111320
radius_lon = dt_contour_ac["radius"].values /(111320 * np.cos(np.deg2rad(dt_contour_ac["lat"].values)))
    
#add new radius_lat/lon to dt_contour
dt_contour_ac = dt_contour_ac.assign(
    radius_lat=("obs", radius_lat),
    radius_lon=("obs", radius_lon))

# Ocean grid
lat_vals = da_ocean_sst["lat"].values
lon_vals = da_ocean_sst["lon"].values

# Eddy properties
eddy_lat = dt_contour_ac["lat"].values
eddy_lon = dt_contour_ac["lon"].values
eddy_time = dt_contour_ac["time"].values
radius_lat = dt_contour_ac["radius_lat"].values * 3 
radius_lon = dt_contour_ac["radius_lon"].values * 3

# Process eddies grouped by unique time
unique_times = np.unique(eddy_time)

keep_eddy = np.ones(n_obs_ac, dtype=bool)

for t in unique_times:
    # Select ocean data for this time only
    da_ocean_time = da_ocean_sst.sel(time=t, method="nearest").values

    # Indices of eddies at this time
    idx_time = np.where(eddy_time == t)[0]

    for i in idx_time:
        lat0, lon0 = eddy_lat[i], eddy_lon[i]
        rad_lat_i, rad_lon_i = radius_lat[i], radius_lon[i]

       # Local window around eddy
        lat_mask = (lat_vals >= lat0 - rad_lat_i) & (lat_vals <= lat0 + rad_lat_i)
        dlon_vals = ((lon_vals - lon0 + 180) % 360) - 180
        lon_mask = np.abs(dlon_vals) <= rad_lon_i
        lat_sub = lat_vals[lat_mask]
        lon_sub = lon_vals[lon_mask]

        lon_grid_sub, lat_grid_sub = np.meshgrid(lon_sub, lat_sub)

        # Ellipse mask
        mask = (dlon(lon_grid_sub, lon0)/rad_lon_i)**2 + ((lat_grid_sub - lat0)/rad_lat_i)**2 <= 1

        var_masked = da_ocean_time[lat_mask, :][:, lon_mask][mask]

        #drop eddies with any NaNs or if mask is empty
        if var_masked.size == 0 or np.isnan(var_masked).any():
            keep_eddy[i] = False

            
dt_contour_ac_clean = dt_contour_ac.isel(obs=keep_eddy)
n_dropped = np.sum(~keep_eddy)
print(f"Dropped {n_dropped} / {n_obs_ac} eddies due to SST NaNs")


        

Dropped 45018 / 1786171 eddies due to SST NaNs


In [9]:
#drop all ac eddies with NaNs in SST
ids_ac = dt_contour_ac_clean["ID"].values
edso_ac_ocean = edso_ac_0.where(edso_ac_0["ID"].isin(ids_ac), drop=True)

In [10]:
#export data
edso_ac_ocean.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso_ac_3R_0_ocean.nc")

In [11]:
#mask for 3R for c eddies
#starting timer
start_time = time.time()
    
#select variable conc for ocean
da_ocean_sst = da_ocean["to"].squeeze()


def dlon(lon, lon0):
    return ((lon - lon0 + 180) % 360) - 180

#creating new data.frame with only the effective_contour_lat/lon and time of the eddies
dt_contour_c = xr.Dataset(
        {
            "contour_lat": edso_c_0["effective_contour_latitude"],
            "contour_lon": edso_c_0["effective_contour_longitude"],
            "time" : edso_c_0["time"],
            "radius" : edso_c_0["effective_radius"],
            "lat" : edso_c_0["latitude"],
            "lon" : edso_c_0["longitude"],
            "ID" : edso_c_0["ID"]
        }
    )

n_obs_c = dt_contour_c.sizes["obs"]
    
#changing radius in m to lat/lon, using simplification 
# 1° lat = 111.32 km = 111320 m
# 1° lon = 111.32 km * cos(lat of eddy cetnre) = 111320 m * cos(lat of eddy centre)
    
radius_lat = dt_contour_c["radius"].values / 111320
radius_lon = dt_contour_c["radius"].values /(111320 * np.cos(np.deg2rad(dt_contour_c["lat"].values)))
    
#add new radius_lat/lon to dt_contour
dt_contour_c = dt_contour_c.assign(
    radius_lat=("obs", radius_lat),
    radius_lon=("obs", radius_lon))

# Ocean grid
lat_vals = da_ocean_sst["lat"].values
lon_vals = da_ocean_sst["lon"].values

# Eddy properties
eddy_lat = dt_contour_c["lat"].values
eddy_lon = dt_contour_c["lon"].values
eddy_time = dt_contour_c["time"].values
radius_lat = dt_contour_c["radius_lat"].values * 3 
radius_lon = dt_contour_c["radius_lon"].values * 3

# Process eddies grouped by unique time
unique_times = np.unique(eddy_time)

keep_eddy_c = np.ones(n_obs_c, dtype=bool)

for t in unique_times:
    # Select ocean data for this time only
    da_ocean_time = da_ocean_sst.sel(time=t, method="nearest").values

    # Indices of eddies at this time
    idx_time = np.where(eddy_time == t)[0]

    for i in idx_time:
        lat0, lon0 = eddy_lat[i], eddy_lon[i]
        rad_lat_i, rad_lon_i = radius_lat[i], radius_lon[i]

       # Local window around eddy
        lat_mask = (lat_vals >= lat0 - rad_lat_i) & (lat_vals <= lat0 + rad_lat_i)
        dlon_vals = ((lon_vals - lon0 + 180) % 360) - 180
        lon_mask = np.abs(dlon_vals) <= rad_lon_i
        lat_sub = lat_vals[lat_mask]
        lon_sub = lon_vals[lon_mask]

        lon_grid_sub, lat_grid_sub = np.meshgrid(lon_sub, lat_sub)

        # Ellipse mask
        mask = (dlon(lon_grid_sub, lon0)/rad_lon_i)**2 + ((lat_grid_sub - lat0)/rad_lat_i)**2 <= 1

        var_masked = da_ocean_time[lat_mask, :][:, lon_mask][mask]

        #drop eddies with any NaNs or if mask is empty
        if var_masked.size == 0 or np.isnan(var_masked).any():
            keep_eddy_c[i] = False

            
dt_contour_c_clean = dt_contour_c.isel(obs=keep_eddy_c)
n_dropped = np.sum(~keep_eddy_c)
print(f"Dropped {n_dropped} / {n_obs_c} eddies due to SST NaNs")


        

Dropped 65335 / 1726939 eddies due to SST NaNs


In [12]:
#drop all c eddies with NaNs as SST
ids_c = dt_contour_c_clean["ID"].values
edso_c_ocean = edso_c_0.where(edso_c_0["ID"].isin(ids_c), drop=True)

In [13]:
#export data
edso_c_ocean.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso_c_3R_0_ocean.nc")